# rxembed embedding options — a KISS playground

**The mental model.** We *never* tell the embedder "put atom *i* at (x, y, z)". We tell it
**relationships between atoms** — "*i* and *j* should be 1.9 Å apart", "the angle *i*-*j*-*k* is ~90°".
Those live in a **bounds matrix**: one `N x N` table of `(min, max)` distances. It is a set of *internal*
coordinates — frame-independent, no origin, no orientation. The embedder's whole job is to find 3D
coordinates consistent with that table.

This notebook plays with the two ways to feed constraints into RDKit's distance-geometry embed:

1. **the bounds matrix** — edit `(min, max)` windows (what rxembed uses)
2. **coordMap** — pin atoms to absolute Cartesian points (what racerts/aqme default to)

...then looks at where **metal–ligand distances** come from, the **embed → graft → optimise** flow,
and finishes with the **KISS verdict**: given coordMap *can* be made to work, what actually wins?

In [1]:
import math
import sys

sys.path.insert(0, "../src")  # run against the repo without installing
import numpy as np
from rdkit import Chem, DistanceGeometry
from rdkit.Chem import AllChem, rdDistGeom
from rdkit.Geometry import Point3D


def dist(conf, i, j):
    p, q = conf.GetAtomPosition(i), conf.GetAtomPosition(j)
    return math.dist((p.x, p.y, p.z), (q.x, q.y, q.z))


def smooth(bm, max_tol=0.4):
    """Triangle-smooth in place; grow tolerance until the matrix is metric-consistent."""
    back = bm.copy()
    tol = 0.0
    while not DistanceGeometry.DoTriangleSmoothing(bm, tol):
        tol = 1.2 * tol + 0.02
        if tol > max_tol:
            return False
        bm[:] = back
    return True


def params(seed=0xF00D):
    p = rdDistGeom.ETKDGv3()
    p.randomSeed = seed
    p.numThreads = 0
    p.pruneRmsThresh = -1.0
    return p

## 1 · The bounds matrix *is* internal coordinates

`GetMoleculeBoundsMatrix` returns RDKit's knowledge-derived bounds for every atom pair. Convention:
**upper triangle `[i][j]` (i<j) = max distance, lower triangle `[j][i]` = min distance**. A bond has a
tight window; two far atoms have a wide one. No coordinates yet — just how far apart everything is allowed to be.

In [2]:
mol = Chem.AddHs(Chem.MolFromSmiles("OCCN"))  # 4 heavy atoms: O0 C1 C2 N3
bm = rdDistGeom.GetMoleculeBoundsMatrix(mol)
np.set_printoptions(precision=2, suppress=True)
print("bounds among the 4 heavy atoms (upper=max, lower=min):")
print(bm[:4, :4])
print(f"\nO0..N3 is allowed anywhere in {bm[3][0]:.2f}..{bm[0][3]:.2f} A (a flexible chain)")

bounds among the 4 heavy atoms (upper=max, lower=min):
[[0.   1.4  2.42 3.64]
 [1.38 0.   1.52 2.46]
 [2.34 1.5  0.   1.46]
 [2.46 2.38 1.44 0.  ]]

O0..N3 is allowed anywhere in 2.46..3.64 A (a flexible chain)


**Constrain a pair = overwrite its window, then re-smooth.** We pull O0 and N3 to a folded contact.
Triangle smoothing propagates that single edit to *every other* pair so the whole table stays consistent —
this is the step that makes constraints compose globally. Then ETKDG embeds against the edited matrix.

In [3]:
def embed_bounds(mol, pair, lo, hi, n=10, seed=0xF00D):
    """Edit one (min,max) window into the bounds matrix and embed n conformers."""
    mol = Chem.Mol(mol)
    bm = rdDistGeom.GetMoleculeBoundsMatrix(mol)
    i, j = pair
    a, b = (i, j) if i < j else (j, i)
    bm[a][b], bm[b][a] = hi, lo  # upper=max, lower=min
    if not smooth(bm):
        return mol, [], "bounds infeasible (triangle smoothing failed)"
    p = params(seed)
    p.SetBoundsMat(bm)
    p.useRandomCoords = False
    ids = list(rdDistGeom.EmbedMultipleConfs(mol, n, p))
    return mol, ids, ""


for lo, hi in [(2.6, 3.0), (2.9, 4.5)]:  # a tight window vs a loose one
    m, ids, err = embed_bounds(mol, (0, 3), lo, hi)
    ds = [dist(m.GetConformer(c), 0, 3) for c in ids]
    print(f"window ({lo},{hi}): {len(ids)} embedded, realised O0..N3 = {min(ds):.2f}..{max(ds):.2f} A")

window (2.6,3.0): 10 embedded, realised O0..N3 = 2.61..3.25 A
window (2.9,4.5): 10 embedded, realised O0..N3 = 2.71..3.66 A


## 1b · Angles live in the distance matrix too (the law of cosines)

A bounds matrix holds **only pairwise distances** — there is no slot for an angle. But an angle *i*–*j*–*k*
(vertex *j*) is fixed once you fix the triangle's three sides. Given the two **leg** lengths *d(i,j)* and
*d(j,k)*, the angle θ at *j* maps one-to-one to the **1–3 distance** *d(i,k)*:

$$d_{ik} = \sqrt{d_{ij}^2 + d_{jk}^2 - 2\,d_{ij}\,d_{jk}\cos\theta}$$

cos is monotonic on [0°, 180°], so a wider angle → a longer 1–3 distance. An angle *window* (θ_lo, θ_hi)
therefore becomes a **distance window** on the (i, k) pair — just another row. This is exactly
`embed/bounds.py::_law_of_cosines`, called for every entry in `cons.angles`.

The legs are taken from whatever the matrix already knows: an explicit distance constraint on that leg if you
gave one, otherwise the midpoint of its current bounds (`leg()` / `_mid`). For a real bond angle the legs
*are* the bond lengths, which the knowledge matrix already pins tightly.

In [4]:
def loc(dij, djk, theta_deg):  # law of cosines: legs + angle -> the 1-3 distance
    return math.sqrt(dij**2 + djk**2 - 2 * dij * djk * math.cos(math.radians(theta_deg)))


# (1) monotonic map: for fixed O-C (1.43) and C-C (1.52) legs, angle -> 1-3 distance
print("angle O-C1-C2  ->  O..C2 (1-3) distance, legs 1.43 & 1.52 A:")
for th in (90, 100, 109, 120, 180):
    print(f"   {th:3d} deg -> {loc(1.43, 1.52, th):.3f} A")


def measure_angle(conf, i, j, k):
    P = conf.GetPositions()
    a, b = P[i] - P[j], P[k] - P[j]
    return math.degrees(math.acos(a @ b / (np.linalg.norm(a) * np.linalg.norm(b))))


# (2) bias angle O0-C1-C2 to 106-112 deg by writing the 1-3 distance window into the matrix
mol = Chem.AddHs(Chem.MolFromSmiles("OCCN"))
bm = rdDistGeom.GetMoleculeBoundsMatrix(mol)
dij = 0.5 * (bm[0][1] + bm[1][0])  # O0-C1 leg  (midpoint of its bounds = the bond)
djk = 0.5 * (bm[1][2] + bm[2][1])  # C1-C2 leg
lo, hi = loc(dij, djk, 106), loc(dij, djk, 112)
bm[0][2], bm[2][0] = hi, lo  # the 1-3 pair (0,2): upper=max, lower=min
smooth(bm)
p = params()
p.SetBoundsMat(bm)
p.useRandomCoords = False
ids = list(rdDistGeom.EmbedMultipleConfs(mol, 12, p))
angs = [measure_angle(mol.GetConformer(c), 0, 1, 2) for c in ids]
print(f"\nlegs O-C1={dij:.3f}  C1-C2={djk:.3f}  ->  1-3 window ({lo:.3f}, {hi:.3f}) A")
print(
    f"target 106-112 deg  ->  realised O0-C1-C2 = {min(angs):.1f}..{max(angs):.1f} deg  (biased; UFF angle term tightens it)"
)

angle O-C1-C2  ->  O..C2 (1-3) distance, legs 1.43 & 1.52 A:
    90 deg -> 2.087 A
   100 deg -> 2.261 A
   109 deg -> 2.402 A
   120 deg -> 2.555 A
   180 deg -> 2.950 A

legs O-C1=1.394  C1-C2=1.514  ->  1-3 window (2.323, 2.412) A
target 106-112 deg  ->  realised O0-C1-C2 = 105.1..114.0 deg  (biased; UFF angle term tightens it)


The realised distances land *near* the window but not exactly inside it — **the bounds matrix only
*biases* the embed**. That is by design: the embed is a *seed*; a restrained-UFF pass (Part 4) tightens it.

## 2 · coordMap — the alternative, its limitation, and the workaround

coordMap pins chosen atoms to **absolute Cartesian points**. Tighter for one rigid template. It appears to
"not work with multiple fragments" — but that warning is really about RDKit's default
`embedFragmentsSeparately=True` (each fragment gets its own frame, so a cross-fragment pin has no shared
frame). Flip that flag and cross-fragment pins *do* work. Let's show all three.

In [5]:
def embed_cmap(mol, cmap, n=8, **kw):
    m = Chem.Mol(mol)
    p = params()
    p.SetCoordMap(cmap)
    for k, v in kw.items():
        setattr(p, k, v)
    return m, list(rdDistGeom.EmbedMultipleConfs(m, n, p))


# (a) single fragment: pin O0, N3 exactly 2.8 A apart -> tight, works out of the box
m, ids = embed_cmap(mol, {0: Point3D(0.0, 0.0, 0.0), 3: Point3D(2.8, 0.0, 0.0)})
ds = [dist(m.GetConformer(c), 0, 3) for c in ids]
print(
    f"(a) single fragment                         : {len(ids)} embedded, O0..N3 = {min(ds):.2f}..{max(ds):.2f} A (tight)"
)

# (b) two fragments, ONE atom pinned in each -> a cross-fragment relationship
frag = Chem.AddHs(Chem.MolFromSmiles("CC(=O)O.CCN"))  # heavy: 0-3 | 4-6
cmap2 = {1: Point3D(0.0, 0.0, 0.0), 6: Point3D(2.7, 0.0, 0.0)}
for tag, kw in [
    ("default (frags separate)", {}),
    ("embedFragmentsSeparately=False", dict(embedFragmentsSeparately=False)),
]:
    m, ids = embed_cmap(frag, cmap2, **kw)
    ds = [dist(m.GetConformer(c), 1, 6) for c in ids] if ids else [float("nan")]
    print(f"(b) two frags, {tag:32s}: {len(ids)} embedded, C1..N6 = {min(ds):.2f}..{max(ds):.2f} A (target 2.70)")

# (c) two fragments, ALL atoms pinned to exact points from a RANDOM placement -> brittle
tmp = Chem.Mol(frag)
AllChem.EmbedMolecule(tmp, randomSeed=2)
allpos = tmp.GetConformer().GetPositions()
cmap_all = {i: Point3D(*allpos[i]) for i in range(frag.GetNumAtoms()) if frag.GetAtomWithIdx(i).GetAtomicNum() > 1}
m, ids = embed_cmap(frag, cmap_all, n=4, embedFragmentsSeparately=False)
print(f"(c) two frags, ALL atoms exact-pinned       : {len(ids)} embedded  <-- exact pins can break smoothing")

(a) single fragment                         : 8 embedded, O0..N3 = 2.63..2.94 A (tight)
(b) two frags, default (frags separate)        : 8 embedded, C1..N6 = 0.83..1.14 A (target 2.70)
(b) two frags, embedFragmentsSeparately=False  : 8 embedded, C1..N6 = 2.74..2.75 A (target 2.70)
(c) two frags, ALL atoms exact-pinned       : 0 embedded  <-- exact pins can break smoothing


[14:05:08] Constrained conformer generation (via the coordMap argument) does not work with molecules that have multiple fragments.
[14:05:08] Could not triangle bounds smooth molecule.


So (b) proves the fragment limit is *dodgeable*. But note what coordMap still can't do: it pins **points
only** — there is no way to say a soft *window* (an NCI 2.6–3.0 Å contact) or an *angle*, and (c) shows that
forcing *exact* absolute coords can leave triangle smoothing no slack. Those are the constraints that
actually make up a metal / TS / NCI problem.

## 3 · Where metal–ligand distances come from

Metal bonds get **distance windows in the same bounds matrix** — no special mechanism (`constraints/metal.py`):

* **from an input geometry** (`.xyz`): the *realised* M–donor length, tight `+/- 0.1 A`.
* **from SMILES** (no geometry): a **covalent-radius sum** — `0.9 x (r_M + r_donor)` for a held spectator
  sphere, `r_M + r_donor` for a fresh dative bond.

Angles come from an ideal-polyhedron table (`metal.ANGLES`) as windows. **All of these are just more rows.**

In [6]:
pt = Chem.GetPeriodicTable()
print("metal-donor distance windows from covalent radii:")
for m_, d_ in [("Pd", "N"), ("Ru", "C"), ("Fe", "C"), ("Mn", "H")]:
    s = pt.GetRcovalent(m_) + pt.GetRcovalent(d_)
    print(
        f"  {m_}-{d_}: covalent sum={s:.2f}  spectator 0.9x=({0.9 * s - 0.05:.2f},{0.9 * s + 0.05:.2f})  "
        f"dative=({s - 0.2:.2f},{s + 0.15:.2f})"
    )
print(
    "\nThese STACK with TS / NCI / angle rows in one matrix -- coordMap could pin none of them (no coords, no windows)."
)

metal-donor distance windows from covalent radii:
  Pd-N: covalent sum=2.10  spectator 0.9x=(1.84,1.94)  dative=(1.90,2.25)
  Ru-C: covalent sum=2.22  spectator 0.9x=(1.95,2.05)  dative=(2.02,2.37)
  Fe-C: covalent sum=2.08  spectator 0.9x=(1.82,1.92)  dative=(1.88,2.23)
  Mn-H: covalent sum=1.70  spectator 0.9x=(1.48,1.58)  dative=(1.50,1.85)

These STACK with TS / NCI / angle rows in one matrix -- coordMap could pin none of them (no coords, no windows).


## 4 · The full flow: embed (bias) → graft exact → optimise (pin)

For a **frozen core** we don't trust the embed (DG approximates a rigid core to ~0.2–0.4 Å — wrong for a
forming bond). We **Kabsch-fit the exact input coordinates onto each embedded pose** (rigid-body, no
distortion), then hold with `AddFixedPoint` (0 DOF) while UFF relaxes the periphery. This is *coordMap-freeze
done right*: same exact core, but robust (no smoothing failures) and frame-free (so it still stacks).

In [7]:
from rxembed.constraints.base import Constraints, add_pairwise_shape
from rxembed.embed.bounds import embed as rx_embed

mol = Chem.AddHs(Chem.MolFromSmiles("c1ccccc1C(=O)OCCN"))
AllChem.EmbedMolecule(mol, randomSeed=1)  # a 'truth' geometry
core = list(range(6))  # freeze the benzene ring
ref = mol.GetConformer().GetPositions()[core]  # its exact coords

cons = Constraints()
add_pairwise_shape(cons, core, mol.GetConformer().GetPositions(), 0.05)  # shape-bias (WINDOWS) in the matrix
cons.frozen |= set(core)
m2 = Chem.Mol(mol)
m2.RemoveAllConformers()
ids = rx_embed(m2, cons, 5, seed=7)


def core_int_rmsd(P):  # frame-free: compare pairwise distances
    def pd(X):
        return np.array([np.linalg.norm(X[i] - X[j]) for i in range(len(X)) for j in range(i + 1, len(X))])

    return float(np.sqrt(np.mean((pd(P) - pd(ref)) ** 2)))


pre = np.mean([core_int_rmsd(m2.GetConformer(c).GetPositions()[core]) for c in ids])
print(f"core internal RMSD after biased embed : {pre:.3f} A   (DG only approximates)")

ca = ref.mean(0)
core0 = ref - ca  # Kabsch graft exact coords onto each pose
for c in ids:
    conf = m2.GetConformer(c)
    emb = conf.GetPositions()[core]
    cb = emb.mean(0)
    u, _, vt = np.linalg.svd(core0.T @ (emb - cb))
    r = vt.T @ np.diag([1, 1, np.sign(np.linalg.det(vt.T @ u.T))]) @ u.T
    for i, p in zip(core, core0 @ r.T + cb):
        conf.SetAtomPosition(int(i), p.tolist())
post = np.mean([core_int_rmsd(m2.GetConformer(c).GetPositions()[core]) for c in ids])
print(f"core internal RMSD after Kabsch graft : {post:.6f} A   (now EXACT; then pinned in optimise)")

core internal RMSD after biased embed : 0.033 A   (DG only approximates)
core internal RMSD after Kabsch graft : 0.000000 A   (now EXACT; then pinned in optimise)


## 5 · KISS verdict — could we infer positions and just use coordMap?

The tempting idea: express constraints as internal distances, **solve them to Cartesian coordinates**, then
feed those as a coordMap. But *solving internal distances → coordinates* **is distance-geometry embedding**.
So you'd embed once to get positions, then embed *again* with them as a coordMap — two embeds to do one, and
you're back to needing a single consistent frame and known coordinates. It's circular; it buys nothing.

**The win of staying in the bounds matrix: you never leave internal coordinates.**

| | bounds matrix (rxembed) | coordMap |
|---|---|---|
| needs to know absolute positions? | **no** | yes (must pre-solve the geometry) |
| soft windows / ranges? | **yes** | no — points only |
| angles? | **yes** (law of cosines → distance) | no |
| multiple fragments? | **native** | only with `embedFragmentsSeparately=False`, and brittle |
| stack TS + metal + NCI + π at once? | **yes** (all just rows) | no |
| exactness for a rigid core? | Kabsch graft after embed | exact pins (can break smoothing) |

So the simplest robust design is exactly what's shipped: **bounds matrix for the seed** (frame-free,
windowed, stackable — no positions required) **+ Kabsch/FF for exactness**. coordMap only pays off in the
one case where all absolute coords already exist and agree (a single rigid template) — and even there the
graft-after-embed matches it more robustly. There is no rxembed workload where inferring positions to feed
coordMap is simpler or better.